# SQL Queries — `data_engineering` Catalog

Connects to the Databricks SQL warehouse using the `school` profile from `~/.databrickscfg` and runs one query per table across all three medallion layers.

| Layer | Tables |
|---|---|
| Bronze | `brfss_2024`, `medicaid_expansion_status_raw`, `svi_2022_us_county` |
| Silver | `brfss_2024_transformed`, `medicaid_expansion_clean_transformed`, `svi_2022_state_transformed` |
| Gold | `dim_respondent`, `dim_behavior`, `dim_chronic_condition`, `dim_healthcare_access`, `dim_preventive_care`, `dim_location`, `dim_medicaid`, `dim_svi`, `dim_time`, `fact_health_response` |

In [ ]:
import configparser
import os

import pandas as pd
from databricks import sql

# Read credentials from ~/.databrickscfg [school] profile
_cfg = configparser.ConfigParser()
_cfg.read(os.path.expanduser("~/.databrickscfg"))
_profile = _cfg["school"]

HOST      = _profile["host"].replace("https://", "").rstrip("/")
TOKEN     = _profile["token"]
HTTP_PATH = "/sql/1.0/warehouses/8a0ddda80f05d456"

conn = sql.connect(
    server_hostname=HOST,
    http_path=HTTP_PATH,
    access_token=TOKEN,
)

def run_query(sql_text: str) -> pd.DataFrame:
    with conn.cursor() as cur:
        cur.execute(sql_text)
        rows = cur.fetchall()
        cols = [c[0] for c in cur.description]
    return pd.DataFrame(rows, columns=cols)

print(f"Connected to {HOST}")

---
## Bronze Layer

In [ ]:
# bronze.brfss_2024 — raw BRFSS 2024 survey responses loaded from CDC source without transformation
# Samples 20 rows to inspect raw CDC-coded field names before cleaning: state FIPS (_STATE),
# collection period (IYEAR, IMONTH), demographics (SEXVAR, _AGEG5YR, _RACE), self-reported
# general and mental/physical health (GENHLTH, PHYSHLTH, MENTHLTH), binary chronic condition
# flags (CVDINFR4=heart attack, CVDSTRK3=stroke, DIABETE4=diabetes, ADDEPEV3=depression),
# BMI category (_BMI5CAT), and the raked complex survey weight (_LLCPWT) that must be applied
# in all downstream prevalence calculations to produce population-representative estimates.
run_query("""
SELECT
    _STATE,
    IYEAR,
    IMONTH,
    SEXVAR,
    _AGEG5YR,
    _RACE,
    GENHLTH,
    PHYSHLTH,
    MENTHLTH,
    _BMI5CAT,
    CVDINFR4,
    CVDSTRK3,
    DIABETE4,
    ADDEPEV3,
    _LLCPWT
FROM data_engineering.bronze.brfss_2024
LIMIT 20
""")

In [ ]:
# bronze.medicaid_expansion_status_raw — raw Kaiser Family Foundation Medicaid expansion tracker
# Full table loaded directly from KFF's state-level ACA Medicaid expansion dataset showing
# each state's adoption decision and, where applicable, the year coverage took effect.
# This is the policy ground truth used to classify states in all downstream Medicaid analysis;
# non-expanding states serve as the counterfactual in health outcome comparisons.
run_query("""
SELECT *
FROM data_engineering.bronze.medicaid_expansion_status_raw
ORDER BY Location
""")

In [ ]:
# bronze.svi_2022_us_county — CDC Social Vulnerability Index 2022 at county level, pre-aggregation
# Surfaces the 20 most vulnerable counties by overall SVI percentile rank (RPL_THEMES = 1.0 is most
# vulnerable) alongside the four CDC theme subscores: socioeconomic status (RPL_THEME1), household
# composition & disability (RPL_THEME2), minority & language access (RPL_THEME3), and
# housing & transportation (RPL_THEME4). Includes county population, uninsured rate, and poverty
# rate as key social determinants. This county-grain data is later aggregated to state level
# in the silver layer for joining with the state-level BRFSS and Medicaid datasets.
run_query("""
SELECT
    STATE,
    COUNTY,
    FIPS,
    E_TOTPOP,
    RPL_THEMES  AS svi_overall,
    RPL_THEME1  AS svi_socioeconomic,
    RPL_THEME2  AS svi_household_composition,
    RPL_THEME3  AS svi_minority_language,
    RPL_THEME4  AS svi_housing_transport,
    EP_UNINSUR  AS pct_uninsured,
    EP_POV150   AS pct_poverty
FROM data_engineering.bronze.svi_2022_us_county
WHERE RPL_THEMES >= 0
ORDER BY RPL_THEMES DESC
LIMIT 20
""")

---
## Silver Layer

In [ ]:
# silver.brfss_2024_transformed — BRFSS data after cleaning, recoding, and label mapping
# Computes survey-weight-adjusted chronic condition and insurance coverage rates per state.
# Using survey weights (rather than raw counts) is essential for BRFSS: the CDC over-samples
# smaller states and certain demographic groups, so unweighted rates systematically misrepresent
# true population prevalence. Ordering by diabetes rate descending surfaces highest-burden states
# and allows visual comparison with Medicaid expansion and SVI status in later queries.
run_query("""
SELECT
    state_name,
    COUNT(*)                                                                   AS respondents,
    ROUND(SUM(CASE WHEN diabetes     = TRUE THEN survey_weight ELSE 0 END)
          / SUM(survey_weight) * 100, 2)                                       AS diabetes_pct,
    ROUND(SUM(CASE WHEN heart_attack = TRUE THEN survey_weight ELSE 0 END)
          / SUM(survey_weight) * 100, 2)                                       AS heart_attack_pct,
    ROUND(SUM(CASE WHEN stroke       = TRUE THEN survey_weight ELSE 0 END)
          / SUM(survey_weight) * 100, 2)                                       AS stroke_pct,
    ROUND(SUM(CASE WHEN depression   = TRUE THEN survey_weight ELSE 0 END)
          / SUM(survey_weight) * 100, 2)                                       AS depression_pct,
    ROUND(SUM(CASE WHEN has_insurance = 'Yes' THEN survey_weight ELSE 0 END)
          / SUM(survey_weight) * 100, 2)                                       AS insured_pct
FROM data_engineering.silver.brfss_2024_transformed
WHERE state_name IS NOT NULL
GROUP BY state_name
ORDER BY diabetes_pct DESC
""")

In [ ]:
# silver.medicaid_expansion_clean_transformed — Medicaid expansion data after status standardization
# Groups states by expansion decision and shows the range of adoption years within each group.
# The earliest/latest year spread reveals how gradually expansion rolled out among adopting states
# (ACA-compliant expansion started in 2014; holdout states adopted as late as 2023), which matters
# for interpreting health outcome differences — states that expanded earlier had more years to
# realize coverage gains by the time the 2024 BRFSS data was collected.
run_query("""
SELECT
    medicaid_expansion_status,
    COUNT(*)            AS state_count,
    MIN(expansion_year) AS earliest_year,
    MAX(expansion_year) AS latest_year
FROM data_engineering.silver.medicaid_expansion_clean_transformed
GROUP BY medicaid_expansion_status
ORDER BY state_count DESC
""")

In [ ]:
# silver.svi_2022_state_transformed — SVI aggregated from county to state level after transformation
# Ranks all 50 states and DC by overall SVI percentile (0 = least vulnerable, 1 = most vulnerable)
# and exposes all four CDC theme subscores alongside three key social determinants: uninsured rate,
# poverty rate, and unemployment rate. Ordering by svi_overall descending immediately surfaces the
# states with the greatest structural disadvantage, which tend to overlap with non-expanding Medicaid
# states — a relationship tested more rigorously in the gold layer cross-tabulation queries.
run_query("""
SELECT
    state_name,
    state_code,
    ROUND(svi_overall, 4)           AS svi_overall,
    ROUND(svi_socioeconomic, 4)     AS svi_socioeconomic,
    ROUND(svi_household, 4)         AS svi_household,
    ROUND(svi_minority, 4)          AS svi_minority,
    ROUND(svi_housing_transport, 4) AS svi_housing_transport,
    ROUND(pct_uninsured, 2)         AS pct_uninsured,
    ROUND(pct_poverty, 2)           AS pct_poverty,
    ROUND(pct_unemployed, 2)        AS pct_unemployed
FROM data_engineering.silver.svi_2022_state_transformed
ORDER BY svi_overall DESC
""")

---
## Gold Layer

In [ ]:
# gold.dim_respondent — demographic dimension: one row per unique sex × age × race × income profile
# Cross-tabulates the four core BRFSS demographic attributes to show how survey respondents
# are distributed across demographic strata. This dimension is joined to fact_health_response
# to enable stratified analysis of disease burden by any combination of demographics.
# High respondent counts indicate the most commonly surveyed profiles and also the groups
# for which prevalence estimates will be most statistically reliable.
run_query("""
SELECT
    sex,
    age_group,
    race,
    income_group,
    COUNT(*) AS respondents
FROM data_engineering.gold.dim_respondent
WHERE sex IS NOT NULL
  AND age_group IS NOT NULL
GROUP BY sex, age_group, race, income_group
ORDER BY respondents DESC
LIMIT 30
""")

In [ ]:
# gold.dim_behavior — behavioral risk factor dimension: one row per unique lifestyle profile
# Cross-tabulates smoking status × BMI category to reveal how behavioral risk factors cluster:
# shows average BMI within each group and the share reporting exercise, binge drinking, and heavy
# drinking. This reveals co-occurrence patterns (e.g., current smokers with obese BMI also tend
# to have lower exercise rates), which are relevant for understanding the behavioral pathways
# through which social vulnerability translates into chronic disease in the fact table analysis.
run_query("""
SELECT
    smoke_status,
    bmi_category_clean,
    COUNT(*)                                                                   AS respondents,
    ROUND(AVG(CAST(bmi_raw AS DOUBLE)), 1)                                     AS avg_bmi,
    ROUND(SUM(CASE WHEN exercise       = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_exercise,
    ROUND(SUM(CASE WHEN binge_drinking = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_binge_drink,
    ROUND(SUM(CASE WHEN heavy_drinker  = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_heavy_drink
FROM data_engineering.gold.dim_behavior
WHERE smoke_status IS NOT NULL
  AND bmi_category_clean IS NOT NULL
GROUP BY smoke_status, bmi_category_clean
ORDER BY respondents DESC
""")

In [ ]:
# gold.dim_chronic_condition — chronic condition dimension grouped by total co-morbidity count
# Breaks down respondents by how many conditions they carry (0 through 5+) and shows the share
# within each tier that has each specific disease. Reveals co-morbidity clustering: respondents
# with 2+ conditions have sharply higher rates of diabetes, depression, and arthritis, reflecting
# the well-documented co-occurrence of chronic diseases. The condition_count grouping also
# serves as a disease-burden index for downstream stratified analysis in the fact table.
run_query("""
SELECT
    CAST(condition_count AS INT)                                               AS num_conditions,
    COUNT(*)                                                                   AS respondents,
    ROUND(SUM(CASE WHEN diabetes     = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_diabetes,
    ROUND(SUM(CASE WHEN heart_attack = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_heart_attack,
    ROUND(SUM(CASE WHEN stroke       = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_stroke,
    ROUND(SUM(CASE WHEN depression   = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_depression,
    ROUND(SUM(CASE WHEN copd         = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_copd,
    ROUND(SUM(CASE WHEN arthritis    = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_arthritis
FROM data_engineering.gold.dim_chronic_condition
GROUP BY CAST(condition_count AS INT)
ORDER BY num_conditions
""")

In [ ]:
# gold.dim_healthcare_access — access dimension stratified by insurance type × PCP status
# Shows cost barrier rates and annual checkup rates across the full insurance type × primary
# care provider (PCP) matrix. The key finding this query surfaces: respondents without insurance
# or a regular doctor face dramatically higher cost barriers and far lower checkup rates,
# directly quantifying how structural access gaps translate into foregone preventive care —
# the mechanism through which Medicaid expansion is hypothesized to improve health outcomes.
run_query("""
SELECT
    has_insurance,
    insurance_type,
    has_pcp,
    COUNT(*)                                                                   AS respondents,
    ROUND(SUM(CASE WHEN cost_barrier  = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_cost_barrier,
    ROUND(SUM(CASE WHEN last_checkup = 'Within past year'
                   THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_checkup_last_year
FROM data_engineering.gold.dim_healthcare_access
WHERE has_insurance IS NOT NULL
GROUP BY has_insurance, insurance_type, has_pcp
ORDER BY respondents DESC
""")

In [ ]:
# gold.dim_preventive_care — preventive care dimension reporting population-level screening uptake
# Returns a single aggregate row with nationwide uptake rates for six preventive services: flu
# and pneumococcal vaccination, mammogram (within 2 years), cervical cancer screening, colorectal
# cancer screening, and HIV testing. These national baselines serve as benchmarks for subgroup
# comparisons — e.g., testing whether states with higher SVI or no Medicaid expansion have
# significantly lower preventive care utilization rates in the fact table analysis.
run_query("""
SELECT
    COUNT(*)                                                                   AS total_respondents,
    ROUND(SUM(CASE WHEN flu_shot          = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_flu_shot,
    ROUND(SUM(CASE WHEN pneumo_vaccine    = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_pneumo_vaccine,
    ROUND(SUM(CASE WHEN mammogram_2yr     = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_mammogram,
    ROUND(SUM(CASE WHEN cervical_screen   = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_cervical_screen,
    ROUND(SUM(CASE WHEN colorectal_screen = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_colorectal_screen,
    ROUND(SUM(CASE WHEN hiv_test          = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_hiv_test
FROM data_engineering.gold.dim_preventive_care
""")

In [ ]:
# gold.dim_location — geographic dimension serving as the bridge key across all three data sources
# Lists every state and territory with its FIPS code, two-letter abbreviation, and full name.
# This dimension is critical in the star schema: BRFSS respondents are keyed by state FIPS,
# SVI data is keyed by state_code (abbreviation), and Medicaid data is keyed by state_name —
# dim_location normalizes all three into a single surrogate key (location_key) that enables
# clean joins across the fact table and all policy/vulnerability dimensions.
run_query("""
SELECT
    location_key,
    state_fips,
    state_code,
    state_name
FROM data_engineering.gold.dim_location
ORDER BY state_name
""")

In [ ]:
# gold.dim_medicaid — Medicaid expansion policy dimension with ACA adoption timeline
# Groups states by expansion decision and year, collecting the list of states in each bucket.
# ACA-compliant expansion started in January 2014; states that adopted later or chose not to
# expand represent deliberate policy decisions with measurable downstream effects on insurance
# coverage and access that are observable when this dimension is joined to the fact table.
# COLLECT_LIST provides a human-readable roster of states in each policy category.
run_query("""
SELECT
    medicaid_expansion_status,
    expansion_year,
    COUNT(*)                 AS state_count,
    COLLECT_LIST(state_name) AS states
FROM data_engineering.gold.dim_medicaid
GROUP BY medicaid_expansion_status, expansion_year
ORDER BY medicaid_expansion_status, expansion_year
""")

In [ ]:
# gold.dim_svi — Social Vulnerability Index dimension at state level, all four CDC themes exposed
# Ranks states by overall SVI score (0 = least vulnerable, 1 = most vulnerable) and exposes
# each of the four CDC theme subscores: socioeconomic status, household composition/disability,
# minority/language access, and housing/transportation. Additional social determinant columns
# (uninsured rate, poverty rate, unemployment, education, disability, minority share) enable
# targeted analysis of which structural factors correlate most strongly with poor health outcomes
# when this dimension is joined to the fact table alongside the Medicaid expansion dimension.
run_query("""
SELECT
    state_name,
    state_code,
    svi_overall,
    svi_socioeconomic,
    svi_household,
    svi_minority,
    svi_housing_transport,
    pct_uninsured,
    pct_poverty,
    pct_unemployed,
    pct_no_hs_diploma,
    pct_disabled,
    pct_minority
FROM data_engineering.gold.dim_svi
ORDER BY svi_overall DESC
""")

In [ ]:
# gold.dim_time — survey time dimension recording interview year, month, and quarter
# Counts distinct month × quarter × year combinations present in the survey data.
# BRFSS is conducted continuously throughout the year rather than in a single wave,
# so this dimension enables time-series analysis of whether response volumes or self-reported
# health metrics shift across months or quarters — a pattern explored in depth in the
# rolling-average and LAG-based time-series query in the Advanced Analytics section below.
run_query("""
SELECT
    interview_year,
    quarter,
    interview_month,
    COUNT(*) AS time_periods
FROM data_engineering.gold.dim_time
GROUP BY interview_year, quarter, interview_month
ORDER BY interview_year, interview_month
""")

In [ ]:
# gold.fact_health_response — star schema fact table joined across six dimension tables
# The central analytical query: joins fact_health_response to dim_location, dim_medicaid,
# dim_svi, dim_chronic_condition, dim_healthcare_access, and dim_behavior to compute weighted
# health outcome rates (diabetes, heart attack, depression, insurance, cost barriers, exercise)
# broken out by Medicaid expansion status × SVI vulnerability quartile. This 2×4 cross-tab
# directly tests whether states that expanded Medicaid show lower disease burden and better
# access, and whether any effect varies by underlying social vulnerability level — linking
# health policy, social determinants, and survey-measured outcomes in a single result set.
run_query("""
SELECT
    m.medicaid_expansion_status,
    CASE
        WHEN s.svi_overall < 0.25 THEN '1 - Low Vulnerability'
        WHEN s.svi_overall < 0.50 THEN '2 - Moderate Vulnerability'
        WHEN s.svi_overall < 0.75 THEN '3 - High Vulnerability'
        ELSE                           '4 - Very High Vulnerability'
    END                                                                        AS svi_quartile,
    COUNT(*)                                                                   AS respondents,
    ROUND(SUM(CAST(f.survey_weight AS DOUBLE)), 0)                            AS weighted_population,
    ROUND(SUM(CASE WHEN c.diabetes     = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
          / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                    AS diabetes_pct,
    ROUND(SUM(CASE WHEN c.heart_attack = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
          / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                    AS heart_attack_pct,
    ROUND(SUM(CASE WHEN c.depression   = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
          / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                    AS depression_pct,
    ROUND(SUM(CASE WHEN a.has_insurance = 'Yes' THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
          / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                    AS insured_pct,
    ROUND(SUM(CASE WHEN a.cost_barrier  = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
          / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                    AS cost_barrier_pct,
    ROUND(SUM(CASE WHEN b.exercise      = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
          / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                    AS exercise_pct
FROM      data_engineering.gold.fact_health_response  f
JOIN      data_engineering.gold.dim_location          l  ON f.location_key  = l.location_key
JOIN      data_engineering.gold.dim_medicaid          m  ON l.state_name    = m.state_name
JOIN      data_engineering.gold.dim_svi               s  ON l.state_code    = s.state_code
JOIN      data_engineering.gold.dim_chronic_condition c  ON f.condition_key = c.condition_key
JOIN      data_engineering.gold.dim_healthcare_access a  ON f.access_key    = a.access_key
JOIN      data_engineering.gold.dim_behavior          b  ON f.behavior_key  = b.behavior_key
WHERE     m.medicaid_expansion_status IS NOT NULL
GROUP BY  m.medicaid_expansion_status, svi_quartile
ORDER BY  m.medicaid_expansion_status, svi_quartile
""")

---
## Advanced Analytics

The following queries demonstrate CTEs, window functions, and time-based analysis against the gold layer.

| Query | Techniques |
|---|---|
| Multi-source state health profile | CTEs, multi-dimension joins, weighted aggregations |
| State diabetes rankings | Window functions: `RANK`, `AVG OVER PARTITION BY`, `NTILE` |
| Monthly survey trends | Time-based analysis, window functions: `LAG`, cumulative `SUM`, rolling `AVG` |

In [ ]:
# CTEs — multi-source state health profile combining outcomes, social context, and policy data
# Uses three named CTEs to modularize a query that would otherwise require a single unwieldy
# join chain: (1) state_health_metrics aggregates weighted disease rates and insurance coverage
# per state by joining the fact table to dim_location, dim_chronic_condition, and dim_healthcare_access;
# (2) state_social_context pulls SVI scores and social determinant percentages from dim_svi;
# (3) state_policy retrieves Medicaid expansion status and adoption year from dim_medicaid.
# The final SELECT assembles all three CTEs into a comprehensive per-state profile ordered by
# diabetes burden, making it easy to see at a glance whether high-disease states also have high
# poverty, high SVI scores, and whether they chose to expand Medicaid.
run_query("""
WITH state_health_metrics AS (
    SELECT
        l.state_name,
        l.state_code,
        COUNT(*)                                                                          AS respondents,
        ROUND(SUM(CAST(f.survey_weight AS DOUBLE)), 0)                                   AS weighted_n,
        ROUND(SUM(CASE WHEN c.diabetes     = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS diabetes_pct,
        ROUND(SUM(CASE WHEN c.heart_attack = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS heart_attack_pct,
        ROUND(SUM(CASE WHEN c.depression   = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS depression_pct,
        ROUND(SUM(CASE WHEN a.has_insurance = 'Yes' THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS insured_pct,
        ROUND(SUM(CASE WHEN a.cost_barrier  = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS cost_barrier_pct
    FROM      data_engineering.gold.fact_health_response  f
    JOIN      data_engineering.gold.dim_location          l ON f.location_key  = l.location_key
    JOIN      data_engineering.gold.dim_chronic_condition c ON f.condition_key = c.condition_key
    JOIN      data_engineering.gold.dim_healthcare_access a ON f.access_key    = a.access_key
    GROUP BY  l.state_name, l.state_code
),
state_social_context AS (
    SELECT
        state_code,
        svi_overall,
        pct_uninsured,
        pct_poverty,
        pct_unemployed
    FROM data_engineering.gold.dim_svi
),
state_policy AS (
    SELECT
        state_name,
        medicaid_expansion_status,
        expansion_year
    FROM data_engineering.gold.dim_medicaid
)
SELECT
    h.state_name,
    p.medicaid_expansion_status,
    p.expansion_year,
    ROUND(s.svi_overall, 4)                  AS svi_overall,
    h.diabetes_pct,
    h.heart_attack_pct,
    h.depression_pct,
    h.insured_pct,
    h.cost_barrier_pct,
    s.pct_poverty,
    s.pct_uninsured
FROM      state_health_metrics  h
JOIN      state_social_context  s ON h.state_code = s.state_code
JOIN      state_policy          p ON h.state_name = p.state_name
ORDER BY  h.diabetes_pct DESC
""")

In [ ]:
# Window functions — state diabetes rankings within and across Medicaid expansion groups
# Demonstrates four window function patterns over a CTE-derived state outcomes table:
# (1) RANK() OVER() for a national ranking by weighted diabetes rate (rank 1 = highest burden);
# (2) RANK() OVER(PARTITION BY medicaid_expansion_status) for within-group peer ranking so you
# can see which states are outliers relative to their policy peers; (3) AVG() OVER(PARTITION BY)
# for each group's mean as a comparison baseline; and (4) NTILE(4) to assign states to national
# diabetes quartiles. The deviation_from_group_avg column isolates state-specific factors from
# the group-level Medicaid expansion effect, helping distinguish policy impact from confounding.
run_query("""
WITH state_outcomes AS (
    SELECT
        l.state_name,
        m.medicaid_expansion_status,
        ROUND(SUM(CASE WHEN c.diabetes     = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS diabetes_pct,
        ROUND(SUM(CASE WHEN a.has_insurance = 'Yes' THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS insured_pct,
        ROUND(SUM(CASE WHEN a.cost_barrier  = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS cost_barrier_pct
    FROM      data_engineering.gold.fact_health_response  f
    JOIN      data_engineering.gold.dim_location          l ON f.location_key  = l.location_key
    JOIN      data_engineering.gold.dim_medicaid          m ON l.state_name    = m.state_name
    JOIN      data_engineering.gold.dim_chronic_condition c ON f.condition_key = c.condition_key
    JOIN      data_engineering.gold.dim_healthcare_access a ON f.access_key    = a.access_key
    WHERE     m.medicaid_expansion_status IS NOT NULL
    GROUP BY  l.state_name, m.medicaid_expansion_status
)
SELECT
    state_name,
    medicaid_expansion_status,
    diabetes_pct,
    insured_pct,
    cost_barrier_pct,
    RANK()  OVER(ORDER BY diabetes_pct DESC)                                              AS national_diabetes_rank,
    RANK()  OVER(PARTITION BY medicaid_expansion_status ORDER BY diabetes_pct DESC)       AS rank_within_medicaid_group,
    ROUND(AVG(diabetes_pct) OVER(PARTITION BY medicaid_expansion_status), 2)              AS group_avg_diabetes_pct,
    ROUND(diabetes_pct - AVG(diabetes_pct) OVER(PARTITION BY medicaid_expansion_status), 2) AS deviation_from_group_avg,
    NTILE(4) OVER(ORDER BY diabetes_pct DESC)                                             AS national_diabetes_quartile
FROM  state_outcomes
ORDER BY national_diabetes_rank
""")

In [ ]:
# Time-based analysis — monthly survey response trends with rolling averages and LAG comparisons
# Joins fact_health_response to dim_time, dim_chronic_condition, and dim_healthcare_access to
# compute monthly response volumes and weighted health rates across the 2024 survey year.
# Four window functions operate over the monthly CTE: (1) cumulative SUM() OVER() for a running
# response total; (2) LAG() to retrieve the prior month's count; (3) month-over-month percent
# change derived from the LAG (NULLIF guards against divide-by-zero in the first month); and
# (4) a 3-month rolling AVG() diabetes rate that smooths sampling noise to reveal true seasonal
# trends. Together these show whether BRFSS participation or self-reported disease rates shift
# meaningfully across calendar months.
run_query("""
WITH monthly_stats AS (
    SELECT
        t.interview_year,
        t.interview_month,
        t.quarter,
        COUNT(*)                                                                          AS monthly_responses,
        ROUND(SUM(CAST(f.survey_weight AS DOUBLE)), 0)                                   AS weighted_respondents,
        ROUND(SUM(CASE WHEN c.diabetes     = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS monthly_diabetes_pct,
        ROUND(SUM(CASE WHEN a.has_insurance = 'Yes' THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS monthly_insured_pct
    FROM      data_engineering.gold.fact_health_response  f
    JOIN      data_engineering.gold.dim_time              t ON f.time_key      = t.time_key
    JOIN      data_engineering.gold.dim_chronic_condition c ON f.condition_key = c.condition_key
    JOIN      data_engineering.gold.dim_healthcare_access a ON f.access_key    = a.access_key
    GROUP BY  t.interview_year, t.interview_month, t.quarter
)
SELECT
    interview_year,
    interview_month,
    quarter,
    monthly_responses,
    weighted_respondents,
    monthly_diabetes_pct,
    monthly_insured_pct,
    SUM(monthly_responses)  OVER(ORDER BY interview_year, interview_month
                                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)      AS cumulative_responses,
    LAG(monthly_responses)  OVER(ORDER BY interview_year, interview_month)               AS prev_month_responses,
    ROUND(
        (monthly_responses - LAG(monthly_responses) OVER(ORDER BY interview_year, interview_month))
        * 100.0
        / NULLIF(LAG(monthly_responses) OVER(ORDER BY interview_year, interview_month), 0),
    1)                                                                                    AS mom_response_pct_change,
    ROUND(AVG(monthly_diabetes_pct) OVER(ORDER BY interview_year, interview_month
                                          ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2)  AS rolling_3mo_diabetes_pct
FROM  monthly_stats
ORDER BY interview_year, interview_month
""")